# OLAP · 01 · BQ1 · Where is insured loss concentrated?
**Owner: Ar** · Reviewer: R · Decision supported: **where Kestrel buys reinsurance cover** (reinsurance and risk manager)

| Query | Question | OLAP operation |
|---|---|---|
| 1.1 | Loss by hazard family → hazard → state, with subtotals | **Roll-up** over two hierarchies (`ROLLUP`) |
| 1.2 | Which state leads each hazard, and with what share? | **Ranking** within a group (`RANK() OVER (PARTITION BY ...)`) |
| 1.3 | How many events make up half the loss? | **Pareto** (running total window) |
| 1.4 | How big and how volatile is the annual loss? | **Roll-up** to calendar year, then statistics across years |

In [ ]:
%run ./00_olap_setup

### 1.1 · Roll-up: hazard family → hazard group → state
`GROUP BY ROLLUP (a, b, c)` returns every level of the hierarchy at once: (family, group, state), (family, group), (family) and the grand total. `grouping()` tells us which level a row belongs to, which we turn into a readable `level` column.

In [ ]:
q11 = olap("bq1_rollup_hazard_state", """
SELECT level, hazard_family, hazard_group, state_code, events, events_with_loss, loss_m, pct_of_total_loss FROM (
  SELECT CASE grouping_id(hazard_family, hazard_group, state_code)
              WHEN 0 THEN '3 · family > hazard > state'
              WHEN 1 THEN '2 · family > hazard'
              WHEN 3 THEN '1 · family'
              ELSE '0 · grand total' END                                  AS level,
         grouping_id(hazard_family, hazard_group, state_code)            AS gid,
         coalesce(hazard_family, 'All families')                         AS hazard_family,
         coalesce(hazard_group,  'All hazards')                          AS hazard_group,
         coalesce(state_code,    'All states')                           AS state_code,
         count(*)                                                        AS events,
         count(loss_m)                                                   AS events_with_loss,
         round(sum(loss_m), 1)                                           AS loss_m,
         round(100 * sum(loss_m) / (SELECT sum(loss_m) FROM v_event_loss), 1) AS pct_of_total_loss
  FROM v_event_loss
  GROUP BY ROLLUP (hazard_family, hazard_group, state_code))
ORDER BY gid DESC, loss_m DESC
""")

Reading the result: the `1 · family` rows show **Hydrological (flood) and Meteorological (hail, storm, cyclone) hazards carry over 90% of all loss**. Drilling into `2 · family > hazard` separates hail (few events, very large) from storm and cyclone (many events, mostly small).

### 1.2 · Ranking: the leading state for each hazard
`UNK` is the Unknown member: events whose state could not be determined. It is ranked like any state so its size stays visible.

In [ ]:
q12 = olap("bq1_top_state_per_hazard", """
WITH hs AS (
  SELECT hazard_group, state_code, count(*) AS events, sum(loss_m) AS loss_m
  FROM v_event_loss GROUP BY hazard_group, state_code),
ranked AS (
  SELECT *, RANK() OVER (PARTITION BY hazard_group ORDER BY loss_m DESC NULLS LAST) AS state_rank,
         loss_m / sum(loss_m) OVER (PARTITION BY hazard_group)          AS share_of_hazard
  FROM hs)
SELECT hazard_group, state_rank, state_code, events, round(loss_m, 1) AS loss_m, round(100 * share_of_hazard, 1) AS pct_of_hazard_loss
FROM ranked WHERE state_rank <= 3
ORDER BY hazard_group, state_rank
""")

### 1.3 · Pareto: how concentrated is loss in a few events?
Rank every event with a loss value from largest to smallest and keep a running share of the total.

In [ ]:
q13 = olap("bq1_pareto_events", """
WITH ranked AS (
  SELECT event_name, start_date, hazard_group, state_code, loss_m,
         row_number() OVER (ORDER BY loss_m DESC)                                                   AS loss_rank,
         sum(loss_m) OVER (ORDER BY loss_m DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
           / sum(loss_m) OVER ()                                                                    AS cumulative_share,
         count(*) OVER ()                                                                           AS events_with_loss
  FROM v_event_loss WHERE loss_m IS NOT NULL)
SELECT loss_rank, event_name, start_date, hazard_group, state_code, round(loss_m, 1) AS loss_m,
       round(100 * cumulative_share, 1) AS cumulative_pct, round(100 * loss_rank / events_with_loss, 1) AS pct_of_events
FROM ranked ORDER BY loss_rank
""", show=False)
display(spark.table(otname("bq1_pareto_events")).limit(15))

for target in (50, 80):
    n = int(q13[q13.cumulative_pct >= target].loss_rank.min())
    print(f"{target}% of loss comes from the largest {n} of {len(q13)} events with a loss value ({100 * n / len(q13):.0f}%).")

### 1.4 · Roll-up to the year: size and volatility of annual loss
Reinsurance is bought per year, so the useful number is the **annual aggregate** loss. The second query rolls the years up into summary statistics.

In [ ]:
q14 = olap("bq1_annual_aggregate", """
SELECT calendar_year, count(*) AS events, round(sum(loss_m), 1) AS loss_m,
       round(max(loss_m), 1) AS largest_event_m,
       round(100 * max(loss_m) / sum(loss_m), 1) AS largest_event_pct_of_year
FROM v_event_loss GROUP BY calendar_year ORDER BY calendar_year
""")

q14s = olap("bq1_annual_aggregate_stats", f"""
WITH y AS (SELECT calendar_year, sum(loss_m) AS loss_m FROM v_event_loss
           WHERE calendar_year < {SCOPE_END_YEAR} GROUP BY calendar_year)
SELECT count(*)                              AS full_years,
       round(avg(loss_m), 0)                 AS mean_annual_loss_m,
       round(percentile_approx(loss_m, 0.5), 0) AS median_annual_loss_m,
       round(stddev(loss_m), 0)              AS stddev_annual_loss_m,
       round(max(loss_m), 0)                 AS worst_year_loss_m,
       max_by(calendar_year, loss_m)         AS worst_year,
       count_if(loss_m > 2 * avg_all)        AS years_above_twice_mean
FROM y CROSS JOIN (SELECT avg(loss_m) AS avg_all FROM y)
""")
# 2026 is excluded from the statistics because the year is not complete.